# Getting Started with qiskit-trev

This tutorial covers the basics of using qiskit-trev for GPU-accelerated quantum circuit simulation with tensor rings.

## What is qiskit-trev?

qiskit-trev is a Qiskit plugin that simulates quantum circuits using **tensor ring** (periodic Matrix Product State) representations, accelerated by PyTorch on GPU. It provides:

- `TREVSampler` — sample bitstrings from quantum circuits
- `TREVEstimator` — compute expectation values of observables
- `TensorRingModel` — low-level model for variational algorithms

In [ ]:
from qiskit.circuit import QuantumCircuit
from qiskit_trev import TREVSampler

## 1. Building a Quantum Circuit

qiskit-trev works with standard Qiskit `QuantumCircuit` objects. Let's build a simple Bell state circuit.

In [ ]:
# Bell state: (|00> + |11>) / sqrt(2)
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

qc.draw("mpl")

## 2. Sampling with TREVSampler

`TREVSampler` implements Qiskit's `BaseSamplerV2` interface. It computes exact probabilities via tensor ring contraction, then samples bitstrings.

Key parameters:
- `rank` — bond dimension of the tensor ring (higher = more accurate, more memory)
- `device` — `"cpu"` or `"cuda"` for GPU acceleration

In [ ]:
# Use "cuda" if you have a GPU, "cpu" otherwise
sampler = TREVSampler(rank=4, device="cpu", default_shots=10000)

job = sampler.run([qc])
result = job.result()

# Get counts from the BitArray
bit_array = result[0].data.meas
counts = bit_array.get_counts()
print("Counts:", counts)

For a Bell state we expect roughly equal counts for `00` and `11`, with negligible `01` and `10`.

## 3. Using TensorRingModel Directly

For variational algorithms, `TensorRingModel` provides a lower-level interface that wraps a parameterized circuit and an observable into a callable model.

In [ ]:
import torch
from qiskit.quantum_info import SparsePauliOp
from qiskit_trev import TensorRingModel

# RY(theta)|0> — a single-qubit parameterized circuit
qc = QuantumCircuit(1)
qc.ry(0.0, 0)  # placeholder parameter

# Observable: Z operator
observable = SparsePauliOp.from_list([("Z", 1.0)])

model = TensorRingModel(qc, observable, rank=1, device="cpu")

# Evaluate <Z> for different angles
import math
for angle in [0.0, math.pi / 4, math.pi / 2, math.pi]:
    ev = model(torch.tensor([angle]))
    print(f"  theta={angle:.4f}  <Z>={ev.item():.4f}  (expected cos(theta)={math.cos(angle):.4f})")

## 4. Batched Evaluation

`evaluate_batch` evaluates many parameter sets in a single GPU call, which is much faster than looping over `forward()` individually.

In [ ]:
# Evaluate 100 angles at once
angles = torch.linspace(0, 2 * math.pi, 100).unsqueeze(1)  # (100, 1)
evs = model.evaluate_batch(angles)

# Plot <Z> = cos(theta)
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(angles.squeeze().numpy(), evs.numpy(), label="qiskit-trev")
plt.plot(angles.squeeze().numpy(), torch.cos(angles.squeeze()).numpy(), "--", label="cos(theta)")
plt.xlabel("theta")
plt.ylabel("<Z>")
plt.legend()
plt.title("Batched expectation value: RY(theta) with Z observable")
plt.tight_layout()
plt.show()

## 5. Choosing the Rank

The `rank` (bond dimension) controls the trade-off between accuracy and speed/memory:

| Rank | Accuracy | Memory | Use case |
|------|----------|--------|----------|
| 1-4 | Low | Minimal | Quick tests, single-qubit |
| 8-16 | Medium | Moderate | Small circuits (< 10 qubits) |
| 32+ | High | Large | Accurate simulation |

For circuits with entanglement (e.g., CNOT gates), higher rank is needed to capture correlations.

In [ ]:
# Bell state: <ZZ> should be 1.0 — higher rank gives better accuracy
qc_bell = QuantumCircuit(2)
qc_bell.h(0)
qc_bell.cx(0, 1)

obs_zz = SparsePauliOp.from_list([("ZZ", 1.0)])

for rank in [1, 2, 4, 8, 16]:
    m = TensorRingModel(qc_bell, obs_zz, rank=rank, device="cpu")
    ev = m(torch.tensor([]))
    print(f"  rank={rank:2d}  <ZZ>={ev.item():.6f}  (exact=1.0)")